# Week 02｜地震地図をつくる・改良する

授業の最後の30分に始め、このNotebookを宿題として仕上げます。最初は**CSVの一行につき、マーカーを一つ**。そこから自分の地図へ。


## 1｜CSVを用意する

授業サイトから**`eq.csv`**をダウンロードし、このNotebookと同じフォルダに保存します。2026年10月7日時点の地震データです。

もっと新しいデータを使うなら、[USGSのCSVダウンロードページ](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)で **Past Day → All Earthquakes** を選びます。ダウンロードしたCSVの名前に`csv_file`を変えてください。

最初の一度だけ、下のCellで地図の道具`folium`をインストールします。CSVを読む`csv`はPythonに最初から入っています。


In [2]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.


## 2｜CSVを読む

`latitude`は緯度、`longitude`は経度、`mag`はマグニチュード。CSVの値は文字なので、地図に使う緯度・経度は`float()`で数に変えます。


In [4]:
import csv
import folium
csv_file = "eq.csv"

with open(csv_file, encoding="utf-8-sig") as file:
    earthquakes = list(csv.DictReader(file))

print(f"{len(earthquakes)}件のデータを読みました")

1850件のデータを読みました


## 3｜一行につき、マーカーを一つ

実行する前に、どこにどんな点が出るか予想しよう。`for`の中が各地震に対して繰り返されます。最後の`earthquake_map`でNotebookに地図を表示します。


In [10]:
earthquake_map = folium.Map(location=[20, 0], zoom_start=2)

for quake in earthquakes:
    folium.Marker(
        location=[float(quake["latitude"]), float(quake["longitude"])],
        tooltip=f"M {quake['mag']}"
    ).add_to(earthquake_map)

earthquake_map

## Challenge｜この地図を改良しよう

授業では一つ改良を始めます。宿題では**二つ以上の改良**を完成させよう。下のコードCellに、上の地図のコードをコピーして変更します。

| 改良の例 | ヒント |
| --- | --- |
| ベースマップを変える | `folium.Map(..., tiles="CartoDB positron")`。地図の見え方を比べる。 |
| ピンを円にする | `folium.Marker`を`folium.CircleMarker`に変え、`radius=5`、`fill=True`を追加する。 |
| 大きさによって色を変える | `float(quake["mag"])`と`if`で色を選び、円の`color`と`fill_color`に使う。空欄の`mag`は先に除く。 |
| 大きな地震だけ表示する | `if quake["mag"] and float(quake["mag"]) >= 4:`の中でマーカーを作る。 |
| 自分の工夫をする | 地域を絞る、円の大きさを変える、クリックで情報を見せる、など。 |

どの改良も、変更する前に結果を予想してから実行しよう。フィルターを使ったら、条件を満たす地震だけ表示されているか確認する。


In [13]:



import folium
import requests

# ========================================
# 1. 地震マップを作成
# ========================================

earthquake_map = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="OpenStreetMap"
)

# ========================================
# 2. 地震をマグニチュード別に表示
# ========================================

for quake in earthquakes:
    if not quake["mag"]:
        continue

    mag = float(quake["mag"])

    if mag >= 6:
        radius = 12
        color = "red"
    elif mag >= 4:
        radius = 8
        color = "orange"
    elif mag >= 2:
        radius = 5
        color = "green"
    else:
        radius = 3
        color = "blue"

    folium.CircleMarker(
        location=[
            float(quake["latitude"]),
            float(quake["longitude"])
        ],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=1,
        tooltip=f"Magnitude: {mag}"
    ).add_to(earthquake_map)

# ========================================
# 3. プレート境界データを取得
# ========================================

plate_url = (
    "https://raw.githubusercontent.com/fraxen/"
    "tectonicplates/master/GeoJSON/PB2002_boundaries.json"
)

response = requests.get(plate_url, timeout=30)
response.raise_for_status()
plate_data = response.json()

# ========================================
# 4. プレート境界を種類ごとに色分け
# ========================================

plate_layer = folium.FeatureGroup(
    name="プレート境界"
)

# 境界の種類ごとの色
boundary_colors = {
    "subduction": "#e63946",
    "transform": "#457b9d",
    "spreading": "#2a9d8f",
    "default": "#9b5de5"
}

def boundary_style(feature):
    properties = feature.get("properties", {})

    # データ内にある境界の種類を取得
    boundary_type = str(
        properties.get(
            "Type",
            properties.get("type", "")
        )
    ).lower()

    # データの種類に応じて色を決定
    if "subduction" in boundary_type:
        color = boundary_colors["subduction"]
    elif "transform" in boundary_type:
        color = boundary_colors["transform"]
    elif "spreading" in boundary_type:
        color = boundary_colors["spreading"]
    else:
        color = boundary_colors["default"]

    return {
        "color": color,
        "weight": 3,
        "opacity": 0.95
    }

# 白い縁取りで境界を見やすくする
folium.GeoJson(
    plate_data,
    style_function=lambda feature: {
        "color": "white",
        "weight": 6,
        "opacity": 0.9
    }
).add_to(plate_layer)

# 色付きの境界線を重ねる
folium.GeoJson(
    plate_data,
    style_function=boundary_style,
    tooltip=folium.GeoJsonTooltip(
        fields=list(
            plate_data["features"][0]["properties"].keys()
        )
    )
).add_to(plate_layer)

plate_layer.add_to(earthquake_map)

# ========================================
# 5. プレート名を地図に表示
# ========================================

plate_labels = [
    ("太平洋プレート", 25, 160),
    ("北米プレート", 45, -130),
    ("南米プレート", -20, -70),
    ("ナスカプレート", -20, -100),
    ("ココスプレート", 10, -100),
    ("カリブプレート", 17, -75),
    ("アフリカプレート", 0, 20),
    ("ユーラシアプレート", 45, 80),
    ("インド・オーストラリアプレート", -25, 100),
    ("南極プレート", -65, 0),
    ("フィリピン海プレート", 20, 135),
    ("アラビアプレート", 25, 45)
]

label_layer = folium.FeatureGroup(
    name="プレート名"
)

for name, lat, lon in plate_labels:
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 12px;
                font-weight: bold;
                color: #222;
                background: rgba(255,255,255,0.9);
                border: 1px solid #555;
                border-radius: 5px;
                padding: 3px 5px;
                white-space: nowrap;
            ">{name}</div>
            """
        ),
        tooltip=name
    ).add_to(label_layer)

label_layer.add_to(earthquake_map)

# ========================================
# 6. レイヤーの表示・非表示を切り替える
# ========================================

folium.LayerControl(
    collapsed=False
).add_to(earthquake_map)

# ========================================
# 7. 地図を表示
# ========================================

earthquake_map


## 宿題｜自分の地震地図

授業で始めた地図を仕上げます。別のNotebookを作る必要はありません。

- タイトルと、誰に見せる地図かを書く。
- 二つ以上改良し、その理由と地図から気づいたことを書く。
- CSVの出典とダウンロード日を書く（付属CSVなら2026年10月7日時点のデータと明記）。
- 上から全Cellを実行し、地図が表示された状態で保存する。
- **完成した`.ipynb`と使用したCSVをUNIPAに提出する。**

**締切：2026年10月14日(水) 23:59 JST**


### 私の地図

**タイトル：世界で発生した地震の分布**

**誰に見せたい？：地震の発生場所が分かりやすくなるように、地図上に地震の位置を表示し、発生場所の分布を確認できるようにしました。地震が多く発生している地域を見つけやすくするため**

**改良したこと・理由：前のマップでは地震が起きているということしかわからなかったので地震のマグニチュードに合わせて、大きい地震ほど大きな赤いピン、小さい地震ほど小さな青・緑のピンにしました。また地震はプレートのズレで起きることがほとんどなのでプレートのある位置も表示しました。**

**地図から気づいたこと：自信は世界中で起きていて特にプレートが重なっているところでは大きい自信が頻繁に起きています。**

**確かめたこと：プレートが多い地域では地震が多い**

**CSVの出典・ダウンロード日：出典：U.S. Geological Survey（USGS）Earthquake Catalog
URL：https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php
ダウンロード日：2026年10月7日時点のUSGSデータ**
